# PoliMillionaire - BM25 (SimpleWiki + KELM) con Reranking BERT (No LLM)

Questa pipeline usa BM25 come recuperatore iperveloce. Per ogni opzione recupera candidati da SimpleWiki e KELM, poi un **Cross-Encoder BERT** rilegge le coppie `(domanda + opzione, passaggio)` e assegna uno score di rilevanza senza generare testo.

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [3]:
import os
import site

# PyTorch va inizializzato prima di import che possono caricare runtime nativi concorrenti.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

dll_directory_handles = []
env_dir = Path(sys.executable).resolve().parent
for dll_dir in [env_dir / "Library" / "bin", env_dir / "DLLs", env_dir / "Lib" / "site-packages" / "torch" / "lib"]:
    if dll_dir.is_dir():
        os.environ["PATH"] = str(dll_dir) + os.pathsep + os.environ.get("PATH", "")
        dll_directory_handles.append(os.add_dll_directory(str(dll_dir)))

import torch
import numpy as np

from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, retrieve, compact_snippet, RetrievalDecision, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [4]:
WIKI_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_bm25.joblib"
KELM_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "kelm_500k_bm25.joblib"

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
kelm_index = load_retrieval_index(KELM_INDEX_PATH)
indexes = [wiki_index, kelm_index]

resource module not available on Windows


In [ ]:
import re
from sentence_transformers import CrossEncoder

bert_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", max_length=512, device="cpu", local_files_only=True)

LOCAL_STOPWORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or", "the", "to", "was", "were", "with", "what", "which", "who", "when", "where"}

def significant_terms(text):
    return [t for t in re.findall(r"[a-z0-9][a-z0-9_+-]*", str(text).lower()) if t not in LOCAL_STOPWORDS and len(t) > 2]

def passage_for_bert(doc, query, max_chars=1400):
    title = str(doc.get("title") or "")
    text = re.sub(r"\s+", " ", str(doc.get("text") or "")).strip()
    if len(text) <= max_chars:
        return f"{title}: {text}" if title else text

    lowered = text.lower()
    positions = [lowered.find(term) for term in significant_terms(query)]
    positions = [pos for pos in positions if pos >= 0]
    center = min(positions) if positions else 0
    start = max(0, center - max_chars // 3)
    end = min(len(text), start + max_chars)
    passage = text[start:end]
    return f"{title}: {passage}" if title else passage

def choose_con_bert(question, index, top_k_bm25=4):
    """BM25 recupera i candidati, BERT reranka i passaggi per ogni opzione."""
    
    global_query = " ".join([str(question.text), *[str(opt.text) for opt in question.options]])
    global_results = retrieve(index, global_query, top_k=top_k_bm25)
    
    option_scores = []
    for opt in question.options:
        option_query = f"{question.text} {opt.text}"
        candidates = retrieve(index, option_query, top_k=top_k_bm25)
        
        if candidates:
            pairs = [(option_query, passage_for_bert(doc, option_query)) for _, doc in candidates]
            bert_scores = [float(score) for score in bert_model.predict(pairs, batch_size=8, show_progress_bar=False)]
            ranked = sorted(zip(bert_scores, candidates), key=lambda row: row[0], reverse=True)
            top_scores = [score for score, _ in ranked[:3]]
            bert_score = top_scores[0] + 0.10 * float(np.mean(top_scores))
            best_doc = ranked[0][1][1]
            best_bm25_score = ranked[0][1][0]
        else:
            bert_score = float("-inf")
            best_doc = None
            best_bm25_score = 0.0
        
        option_scores.append({
            "option_id": opt.id,
            "option_text": str(opt.text),
            "score": bert_score,
            "evidence_score": bert_score,
            "best_bm25_score": best_bm25_score,
            "top_evidence": compact_snippet(best_doc) if best_doc else None,
        })

    option_scores.sort(key=lambda row: row["score"], reverse=True)
    best = option_scores[0]
    second_score = option_scores[1]["score"] if len(option_scores) > 1 else 0.0
    confidence = best["score"] - second_score

    return RetrievalDecision(
        option_id=int(best["option_id"]),
        option_text=str(best["option_text"]),
        confidence=float(confidence),
        option_scores=option_scores,
        evidence=[{"score": sc, **compact_snippet(dc)} for sc, dc in global_results]
    )

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
ATTEMPTS_PER_COMPETITION = 10
STRATEGY_NAME = "bm25_mixed_plus_bert_reranker"

rows = run_all_competitions(
    client=client,
    index=indexes,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
    choose_fn=choose_con_bert, # <--- Magia: iniettiamo la nostra funzione BERT custom senza toccare lo script master!
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_kelm_bert_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

In [ ]:
summarize(rows)